# 04 – Ayar ve Projenin İki Deneyi
**Hafta:** H9–H10 (14.12 – 27.12) · **Puanlama:** Değerlendirme %20 · **Slaytlar:** 35–38, 42, 48

Her deney aynı yapıda yazılır (slayt 42: **bir seferde tek değişiklik**):
**Soru → Beklenti → Tek değişiklik → Kod → Sonuç → Yorum**

> Test setine hâlâ bakılmaz. Karşılaştırmalar doğrulama setleri üzerinden yapılır.

### Adım 0: Kurulum ve veri
*Slayt 53*

**Ne yapıyoruz:** Yükle, temizle, iki bölmeyi kur.

In [ ]:
# proje kökünü Python yoluna ekle (src/ klasörünü kullanabilmek için)
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold

from src import config
from src.data import load_raw, clean, split_xy, random_split, time_split
from src.features import make_preprocessor, drop_cols
from src.models import PageValuesRule
from src.evaluate import scores, best_threshold, results_table, plot_pr_curves

df = clean(load_raw())
train, val, test = random_split(df)
t_train, t_val, t_test = time_split(df)

### Adım 1: Hiperparametre ayarı
*Slayt 36, 42*

**Ne yapıyoruz:** 03'te seçtiğimiz modelin ayarlarını **sadece eğitim setinde** çapraz doğrulamayla arıyoruz.

**Sözde kod (slayt 57):**
```
aday ayarlar = {...}
arama = GridSearchCV(model, aday ayarlar, scoring='average_precision', cv=5)
arama.fit(X_train, y_train)
en iyi ayarlar -> doğrulamada değerlendir
```

**Kontrol soruları:**
- Neden ayarı doğrulama ya da test setinde değil, eğitim setinde çapraz doğrulamayla yapıyoruz?

In [ ]:
# aday ayarlar

# arama (sadece eğitim verisi)

# en iyi ayarla doğrulamada değerlendir

**Yorum:** 

## Deney 1: PageValues sızıntısı
| | |
|---|---|
| **Soru** | PageValues tahmin anında bilinmeyen bilgi taşıyorsa, model ona ne kadar bağımlı? |
| **Beklenti** | (deneyden önce yazın) |
| **Tek değişiklik** | Özellik listesinden `PageValues` çıkarılıyor. Model, ayarlar, bölme ve eşik yöntemi aynı. |

### Adım 2a: PageValues ile
*Slayt 37–38*

**Ne yapıyoruz:** Adım 1'deki en iyi model, tüm özelliklerle.

**Sözde kod (slayt 57):**
```
X = tüm özellikler
fit(eğitim) -> olasılık(doğrulama) -> eşik seç -> skor
```

In [ ]:
# PageValues ile

### Adım 2b: PageValues olmadan
*Slayt 37–38*

**Ne yapıyoruz:** Aynı model, `drop_cols(config.NUM_COLS, "PageValues")` ile.

**Sözde kod (slayt 57):**
```
X = PageValues hariç özellikler
fit(eğitim) -> olasılık(doğrulama) -> eşik seç -> skor
```

In [ ]:
# PageValues olmadan

### Adım 2c: Karşılaştırma
*Slayt 37–38*

**Ne yapıyoruz:** İki sonuç ve kural baseline'ı yan yana.

**Kontrol soruları:**
- Performans ne kadar düştü? Sıfıra mı indi, yoksa model başka değişkenlerden de öğreniyor mu?
- Gerçek bir sitede oturum sırasında kullanılacaksa PageValues'u kullanmalı mıyız?

In [ ]:
# karşılaştırma tablosu

**Yorum:** 

## Deney 2: Rastgele bölme / zamana göre bölme
| | |
|---|---|
| **Soru** | Model geçmiş aylarla eğitilip sonraki aylarda kullanılırsa ne kadar iyi çalışır? |
| **Beklenti** | (deneyden önce yazın) |
| **Tek değişiklik** | Bölme yöntemi. İki tarafta da `Month` çıkarılır ki fark sadece bölmeden gelsin. |

### Adım 3a: Rastgele bölme (Month olmadan)
*Slayt 33, 35*

**Ne yapıyoruz:** Adil karşılaştırma için `drop_cols(config.CAT_COLS, "Month")`.

**Sözde kod (slayt 57):**
```
X = Month hariç özellikler, rastgele bölme
fit(eğitim) -> olasılık(doğrulama) -> eşik seç -> skor
```

In [ ]:
# rastgele bölme

### Adım 3b: Zamana göre bölme (Month olmadan)
*Slayt 35*

**Ne yapıyoruz:** Eğitim: Şub–Ağu, doğrulama: Eyl–Eki. Eşik zaman bölmesinin doğrulama setinde seçilir.

**Sözde kod (slayt 57):**
```
X = Month hariç özellikler, zaman bölmesi
fit(Şub–Ağu) -> olasılık(Eyl–Eki) -> eşik seç -> skor
```

**Kontrol soruları:**
- Neden bu deneyde `Month` çıkarılmalı?

In [ ]:
# zamana göre bölme

### Adım 3c: Karşılaştırma
*Slayt 35, 41*

**Ne yapıyoruz:** İki bölmenin sonuçları yan yana.

**Kontrol soruları:**
- Hangi bölme gerçek kullanıma daha yakın?
- Basit model ile karmaşık model arasındaki fark iki bölmede aynı mı? (slayt 41)

In [ ]:
# karşılaştırma tablosu

**Yorum:** 

### Adım 4: Final seçimlerini sabitle
*Slayt 34*

**Ne yapıyoruz:** 06'da test setinde kullanılacak seçimleri şimdi yazıyoruz. Test sonucuna bakıp bunlar değiştirilmez.

| Seçim | Değer | Gerekçe |
|---|---|---|
| Model ve ayarlar | | |
| Özellikler (PageValues var mı?) | | |
| Eşik | | |
| Bölme | | |

## Bulgular ve kararlar
> Bu hücreyi **kendi cümlelerinizle** yazın. Sunumda ve hocanın kontrolünde bunu savunacaksınız.

- Bulgu 1:
- Bulgu 2:
- Bu bulgulara göre aldığımız karar(lar):
- Deney günlüğüne yazıldı mı? (`reports/deney_gunlugu.md`)
- Git commit atıldı mı?